# Irish Rent Prices 2020-2025 (RTB Official Data)


Average monthly rent across 26 Irish counties

In [1]:
import pandas as pd
import numpy as np
import io

# First, let's load the data from your file
# Since you've provided the content as text, we'll read it directly

# Read the CSV data
df = pd.read_csv('archive\irish_rent_full.csv')

# Display basic information about the dataset
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
print(df.head())

DATASET OVERVIEW
Shape: (50208, 15)

Columns:
['rent_euro', 'year', 'half', 'half_year', 'time_period', 'county', 'province', 'area', 'location', 'property_type', 'bedrooms', 'bedrooms_num', 'is_dublin', 'is_city', 'is_county_aggregate']

Data Types:
rent_euro              float64
year                     int64
half                     int64
half_year               object
time_period              int64
county                  object
province                object
area                    object
location                object
property_type           object
bedrooms                object
bedrooms_num           float64
is_dublin                 bool
is_city                   bool
is_county_aggregate       bool
dtype: object

First 5 rows:
   rent_euro  year  half half_year  time_period  county  province  \
0     835.90  2020     1    2020H1            1  Carlow  Leinster   
1     860.74  2020     1    2020H1            1  Carlow  Leinster   
2     910.91  2020     1    2020H1            1 

In [2]:
# ============================================================================
# DATA CLEANING
# ============================================================================

print("=" * 80)
print("DATA CLEANING PROCESS")
print("=" * 80)

# 1. Check for missing values
print("\n1. MISSING VALUES CHECK")
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percent
})
print(missing_df[missing_df['Missing Count'] > 0])

# 2. Check for duplicates
print("\n2. DUPLICATES CHECK")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicates. New shape: {df.shape}")

# 3. Check data types and convert if necessary
print("\n3. DATA TYPE VERIFICATION")
print("Current data types:")
print(df.dtypes)

# Convert rent_euro to float if it's not already
if df['rent_euro'].dtype != 'float64':
    df['rent_euro'] = pd.to_numeric(df['rent_euro'], errors='coerce')
    print("Converted rent_euro to float")

# Convert year to int if it's not already
if df['year'].dtype != 'int64':
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    print("Converted year to int")

# Convert bedrooms_num to float (since it has decimal values like 1.5 for "1 to 2 bed")
if df['bedrooms_num'].dtype != 'float64':
    df['bedrooms_num'] = pd.to_numeric(df['bedrooms_num'], errors='coerce')
    print("Converted bedrooms_num to float")

# 4. Check for negative or zero rent values
print("\n4. RENT VALUES CHECK")
negative_rent = df[df['rent_euro'] <= 0]
print(f"Rows with rent <= 0: {len(negative_rent)}")
if len(negative_rent) > 0:
    print("Sample of negative/zero rent rows:")
    print(negative_rent[['rent_euro', 'county', 'year', 'property_type']].head())

# 5. Check categorical columns for consistency
print("\n5. CATEGORICAL COLUMNS CHECK")
categorical_cols = ['county', 'province', 'area', 'location', 'property_type', 'bedrooms']

for col in categorical_cols:
    print(f"\n{col.upper()} - Unique values: {df[col].nunique()}")
    print(f"Sample values: {df[col].dropna().unique()[:10]}")

# 6. Check time period consistency
print("\n6. TIME PERIOD CHECK")
print(f"Years present: {sorted(df['year'].unique())}")
print(f"Half-years present: {sorted(df['half_year'].unique())}")
print(f"Time periods present: {sorted(df['time_period'].unique())}")

# 7. Check boolean columns
print("\n7. BOOLEAN COLUMNS CHECK")
bool_cols = ['is_dublin', 'is_city', 'is_county_aggregate']
for col in bool_cols:
    print(f"{col} - Values: {df[col].unique()}")

# 8. Check for inconsistencies in area vs location
print("\n8. AREA VS LOCATION CHECK")
# Check if area appears within location string
sample = df.sample(min(10, len(df)))
for idx, row in sample.iterrows():
    if pd.notna(row['area']) and pd.notna(row['location']):
        contains = row['area'].lower() in row['location'].lower()
        print(f"Area: '{row['area']}' | Location: '{row['location']}' | Area in location: {contains}")

# 9. Summary statistics for rent
print("\n9. RENT STATISTICS")
print(df['rent_euro'].describe())

# 10. Check for outliers in rent (using IQR method)
Q1 = df['rent_euro'].quantile(0.25)
Q3 = df['rent_euro'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['rent_euro'] < lower_bound) | (df['rent_euro'] > upper_bound)]
print(f"\nPotential outliers (based on IQR): {len(outliers)} rows")
print(f"Lower bound: €{lower_bound:.2f}, Upper bound: €{upper_bound:.2f}")
print(f"Min rent: €{df['rent_euro'].min():.2f}, Max rent: €{df['rent_euro'].max():.2f}")

DATA CLEANING PROCESS

1. MISSING VALUES CHECK
              Missing Count  Missing Percentage
bedrooms_num          12339           24.575765

2. DUPLICATES CHECK
Number of duplicate rows: 0

3. DATA TYPE VERIFICATION
Current data types:
rent_euro              float64
year                     int64
half                     int64
half_year               object
time_period              int64
county                  object
province                object
area                    object
location                object
property_type           object
bedrooms                object
bedrooms_num           float64
is_dublin                 bool
is_city                   bool
is_county_aggregate       bool
dtype: object

4. RENT VALUES CHECK
Rows with rent <= 0: 0

5. CATEGORICAL COLUMNS CHECK

COUNTY - Unique values: 26
Sample values: ['Carlow' 'Cavan' 'Clare' 'Cork' 'Donegal' 'Dublin' 'Galway' 'Kerry'
 'Kildare' 'Kilkenny']

PROVINCE - Unique values: 4
Sample values: ['Leinster' 'Ulster' 'Munste